# Raw Episode Collection

Collect raw Super Mario Bros. level 1-1 episodes (aiming for around 80 episodes for initial smoke-test)

Artifact structure:

```text
data/raw/episodes/ep_000001/
  frames.mkv     # lossless video, preferably FFV1
  steps.jsonl    # one JSON row per frame/action
  meta.json      # episode-level metadata
```

In [1]:
# Initialize the data collection directory structure
from __future__ import annotations

import sys
import gzip
import json
import random
import shutil
from dataclasses import asdict, dataclass, replace
from pathlib import Path
from typing import Any

import numpy as np

try:
    import imageio.v2 as imageio
except ImportError:
    imageio = None

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "data-collection" else Path.cwd()

# Ensure rlab is in python path
RLAB_SRC = REPO_ROOT / "data-collection" / "agent" / "rlab" / "src"
if RLAB_SRC.exists() and str(RLAB_SRC) not in sys.path:
    sys.path.insert(0, str(RLAB_SRC))

from rlab.env import EnvConfig, make_eval_vec_env
from stable_baselines3 import PPO

RAW_EPISODES_DIR = REPO_ROOT / "data" / "raw" / "episodes"
START_STATES_DIR = REPO_ROOT / "data-collection" / "start-states"

print(f"Raw episodes will be written to: {RAW_EPISODES_DIR}")
print(f"Random-start states will be read from: {START_STATES_DIR}")


Raw episodes will be written to: /Users/soheilchavoshi/Projects/mario-world/data/raw/episodes
Random-start states will be read from: /Users/soheilchavoshi/Projects/mario-world/data-collection/start-states


In [2]:
@dataclass
class CollectionConfig:
    game: str = "SuperMarioBros-Nes-v0"
    level: str = "1-1"
    fps: int = 60
    max_steps: int = 4500
    action_set: str = "simple"
    policy_name: str = "ppo_pretrained"
    run_type: str = "normal"  # normal | perturbed | random_start_perturbed

    # Lightweight behavior perturbation knobs.
    use_perturbation: bool = False
    perturbation_mode: str = "dropout"  # dropout | activation_noise
    perturbation_strength: float = 0.05
    action_noise: float = 0.0
    sticky_action_prob: float = 0.0

    # Random-start knobs.
    use_random_start: bool = False
    start_state: str = "level_start"
    start_state_id: str | None = None
    start_states_dir: str = str(START_STATES_DIR)

    overwrite: bool = True


CONFIG = CollectionConfig()
asdict(CONFIG)


{'game': 'SuperMarioBros-Nes-v0',
 'level': '1-1',
 'fps': 60,
 'max_steps': 4500,
 'action_set': 'simple',
 'policy_name': 'ppo_pretrained',
 'run_type': 'normal',
 'use_perturbation': False,
 'perturbation_mode': 'dropout',
 'perturbation_strength': 0.05,
 'action_noise': 0.0,
 'sticky_action_prob': 0.0,
 'use_random_start': False,
 'start_state': 'level_start',
 'start_state_id': None,
 'start_states_dir': '/Users/soheilchavoshi/Projects/mario-world/data-collection/start-states',
 'overwrite': True}

In [3]:
def json_sanitize(obj):
    if isinstance(obj, (np.integer, np.floating, np.bool_)):
        return obj.item()
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {str(k): json_sanitize(v) for k, v in obj.items()}
    elif isinstance(obj, (list, tuple)):
        return [json_sanitize(v) for v in obj]
    elif isinstance(obj, (int, float, str, bool, type(None))):
        return obj
    else:
        return str(obj)


class EpisodeWriter:
    def __init__(self, episode_dir: Path, config: CollectionConfig):
        self.episode_dir = episode_dir
        self.tmp_dir = episode_dir.with_name(episode_dir.name + ".tmp")
        self.config = config
        self.video_writer = None
        self.steps_file = None

    def __enter__(self):
        if self.episode_dir.exists():
            if not self.config.overwrite:
                raise FileExistsError(f"Episode already exists: {self.episode_dir}")
            shutil.rmtree(self.episode_dir)
        if self.tmp_dir.exists():
            shutil.rmtree(self.tmp_dir)
        self.tmp_dir.mkdir(parents=True)

        (self.tmp_dir / "meta.json").write_text(json.dumps(asdict(self.config), indent=2, default=json_sanitize) + "\n")
        self.steps_file = (self.tmp_dir / "steps.jsonl").open("w")

        if imageio is None:
            raise ImportError("Install imageio[ffmpeg] to write frames.mkv")
        self.video_writer = imageio.get_writer(
            self.tmp_dir / "frames.mkv",
            fps=self.config.fps,
            codec="ffv1",
            macro_block_size=None,
        )
        return self

    def write_step(self, t: int, frame: np.ndarray, action: int, reward: float, done: bool, info: dict[str, Any]):
        frame = np.asarray(frame, dtype=np.uint8)
        self.video_writer.append_data(frame)
        row = {
            "t": int(t),
            "action": int(action),
            "reward": float(reward),
            "done": bool(done),
            "info": json_sanitize(info),
        }
        self.steps_file.write(json.dumps(row, default=json_sanitize) + "\n")

    def __exit__(self, exc_type, exc, tb):
        if self.video_writer is not None:
            self.video_writer.close()
        if self.steps_file is not None:
            self.steps_file.close()
        if exc_type is None:
            self.tmp_dir.rename(self.episode_dir)
        else:
            shutil.rmtree(self.tmp_dir, ignore_errors=True)
        return False


In [4]:
def make_env(config: CollectionConfig):
    """Wire this to the current rlab/stable-retro setup."""
    rlab_config = EnvConfig(
        game=config.game,
        env_provider="supermariobrosnes-turbo",
        action_set=config.action_set,
        frame_skip=4,
        observation_size=84,
        obs_crop=(32, 0, 0, 0),
        max_episode_steps=config.max_steps,
    )
    return make_eval_vec_env(rlab_config, n_envs=1, seed=random.randint(0, 1000000))


def load_policy(config: CollectionConfig):
    """Wire this to the pretrained PPO model."""
    model_path = REPO_ROOT / "data-collection" / "agent" / "NES-SuperMarioBros_Level1-1_gray84-hudcrop-stack4-simple_ppo" / "model.zip"
    if not model_path.exists():
        model_path = REPO_ROOT / "training" / "agent" / "NES-SuperMarioBros_Level1-1_gray84-hudcrop-stack4-simple_ppo" / "model.zip"
    return PPO.load(model_path)


def unwrap_env(env):
    curr = env
    while True:
        if hasattr(curr, "venv"):
            curr = curr.venv
        elif hasattr(curr, "envs") and len(curr.envs) > 0:
            curr = curr.envs[0]
        elif hasattr(curr, "env"):
            curr = curr.env
        else:
            break
    return curr


def get_rgb_frame(env, obs) -> np.ndarray:
    """Prefer env.render() if configured for rgb_array; fall back to obs for now."""
    if hasattr(env, "render"):
        frame = env.render(mode="rgb_array") if "mode" in getattr(env.render, "__code__", object()).co_varnames else env.render()
        if frame is not None:
            if isinstance(frame, list) and len(frame) > 0:
                frame = frame[0]
            return np.asarray(frame, dtype=np.uint8)
    return np.asarray(obs, dtype=np.uint8)


def policy_action(policy, obs, deterministic: bool):
    if hasattr(policy, "predict"):
        action, _ = policy.predict(obs, deterministic=deterministic)
        return action
    action = policy(obs)
    if isinstance(action, tuple):
        action = action[0]
    return action


def install_policy_perturbation(policy, config: CollectionConfig):
    """Install simple PyTorch forward hooks on an SB3/PyTorch policy.

    This is intentionally best-effort. If the loaded policy is not PyTorch-shaped,
    it returns no hooks and the remaining perturbation knobs still work.
    """
    if not config.use_perturbation or config.perturbation_strength <= 0:
        return []

    try:
        import torch
        import torch.nn as nn
        import torch.nn.functional as F
    except ImportError:
        return []

    module_root = getattr(policy, "policy", policy)
    if not hasattr(module_root, "modules"):
        return []

    handles = []

    def hook(_module, _inputs, output):
        if not torch.is_tensor(output):
            return output
        if config.perturbation_mode == "activation_noise":
            return output + torch.randn_like(output) * config.perturbation_strength
        return F.dropout(output, p=config.perturbation_strength, training=True)

    for module in module_root.modules():
        if isinstance(module, (nn.Linear, nn.Conv2d)):
            handles.append(module.register_forward_hook(hook))

    return handles


def remove_hooks(handles):
    for handle in handles:
        handle.remove()


def choose_action(env, policy, obs, config: CollectionConfig, last_action: int | None = None) -> int:
    if last_action is not None and random.random() < config.sticky_action_prob:
        return int(last_action)
    if random.random() < config.action_noise:
        action_space = getattr(env, "single_action_space", getattr(env, "action_space", None))
        if action_space is not None:
            return int(action_space.sample())

    action = policy_action(policy, obs, deterministic=not config.use_perturbation)
    return int(np.asarray(action).flat[0])


In [5]:
def list_start_states(config: CollectionConfig) -> list[Path]:
    state_dir = Path(config.start_states_dir)
    if not state_dir.exists():
        return []
    return sorted([*state_dir.glob("*.state"), *state_dir.glob("*.state.gz"), *state_dir.glob("*.bin")])


def choose_start_state(config: CollectionConfig) -> Path:
    states = list_start_states(config)
    if not states:
        raise FileNotFoundError(f"No start states found in {config.start_states_dir}")
    if config.start_state_id is not None:
        matches = [p for p in states if p.stem == config.start_state_id or p.name == config.start_state_id]
        if not matches:
            raise FileNotFoundError(f"No start state matched {config.start_state_id!r}")
        return matches[0]
    return random.choice(states)


def read_state_bytes(path: Path) -> bytes:
    raw = path.read_bytes()
    try:
        return gzip.decompress(raw)
    except OSError:
        return raw


def reset_with_optional_random_start(env, config: CollectionConfig):
    if not config.use_random_start or not list_start_states(config):
        obs_res = env.reset()
        obs = obs_res[0] if isinstance(obs_res, tuple) else obs_res
        info = obs_res[1] if isinstance(obs_res, tuple) and len(obs_res) > 1 else {}
        if isinstance(info, list) and len(info) > 0:
            info = info[0]
        return obs, info, {"start_state": config.start_state}

    state_path = choose_start_state(config)
    base_env = unwrap_env(env)
    state_bytes = read_state_bytes(state_path)

    if hasattr(base_env, "initial_state"):
        base_env.initial_state = state_bytes
        obs_res = env.reset()
    elif hasattr(base_env, "em") and hasattr(base_env.em, "set_state"):
        obs_res = env.reset()
        base_env.em.set_state(state_bytes)
    else:
        obs_res = env.reset()

    obs = obs_res[0] if isinstance(obs_res, tuple) else obs_res
    info = obs_res[1] if isinstance(obs_res, tuple) and len(obs_res) > 1 else {}
    if isinstance(info, list) and len(info) > 0:
        info = info[0]
    return obs, info, {"start_state": state_path.name}


def save_current_start_state(env, name: str) -> Path:
    """Optional helper for later: save the current emulator state as a random-start candidate."""
    base_env = unwrap_env(env)
    if not (hasattr(base_env, "em") and hasattr(base_env.em, "get_state")):
        raise NotImplementedError("This emulator object does not expose em.get_state()")
    START_STATES_DIR.mkdir(parents=True, exist_ok=True)
    path = START_STATES_DIR / f"{name}.state"
    path.write_bytes(gzip.compress(base_env.em.get_state()))
    return path


In [6]:
def run_episode(
    episode_id: int,
    config: CollectionConfig,
    *,
    use_perturbation: bool | None = None,
    use_random_start: bool | None = None,
) -> dict[str, Any]:
    if use_perturbation is not None or use_random_start is not None:
        config = replace(
            config,
            use_perturbation=config.use_perturbation if use_perturbation is None else use_perturbation,
            use_random_start=config.use_random_start if use_random_start is None else use_random_start,
        )

    episode_dir = RAW_EPISODES_DIR / f"ep_{episode_id:06d}"
    env = make_env(config)
    policy = load_policy(config)
    hooks = install_policy_perturbation(policy, config)

    total_reward = 0.0
    last_action = None

    try:
        obs, info, start_meta = reset_with_optional_random_start(env, config)

        with EpisodeWriter(episode_dir, config) as writer:
            for t in range(config.max_steps):
                action = choose_action(env, policy, obs, config, last_action=last_action)
                step_res = env.step([action] if hasattr(env, "num_envs") else action)
                if len(step_res) == 4:
                    obs, reward, done, info_res = step_res
                    reward_val = float(reward[0]) if isinstance(reward, (list, np.ndarray)) else float(reward)
                    done_val = bool(done[0]) if isinstance(done, (list, np.ndarray)) else bool(done)
                    info_dict = info_res[0] if isinstance(info_res, list) and len(info_res) > 0 else info_res
                else:
                    obs, reward_val, terminated, truncated, info_dict = step_res
                    done_val = bool(terminated or truncated)

                total_reward += float(reward_val)
                last_action = action

                frame = get_rgb_frame(env, obs)
                step_info = dict(info_dict or {})
                if t == 0:
                    step_info.update(start_meta)
                writer.write_step(t, frame, action, reward_val, done_val, step_info)

                if done_val:
                    break
    finally:
        remove_hooks(hooks)
        env.close()

    return {"episode_id": episode_id, "steps": t + 1, "reward": total_reward, "dir": str(episode_dir)}


## First Collection Plan

Start with a tiny smoke test before the 80-episode run.

Planned 80-episode mix:

```text
30 normal pretrained-agent episodes
30 perturbed-agent episodes from level start
20 perturbed-agent episodes from saved mid-level states
```

In [7]:
def make_collection_plan(
    *,
    normal: int = 30,
    perturbed: int = 30,
    random_start_perturbed: int = 20,
    base_config: CollectionConfig = CONFIG,
) -> list[CollectionConfig]:
    plan = []

    for _ in range(normal):
        plan.append(replace(base_config, run_type="normal", use_perturbation=False, use_random_start=False))

    for _ in range(perturbed):
        plan.append(
            replace(
                base_config,
                run_type="perturbed",
                use_perturbation=True,
                use_random_start=False,
                perturbation_strength=0.05,
                action_noise=0.02,
                sticky_action_prob=0.05,
            )
        )

    for _ in range(random_start_perturbed):
        plan.append(
            replace(
                base_config,
                run_type="random_start_perturbed",
                use_perturbation=True,
                use_random_start=True,
                perturbation_strength=0.05,
                action_noise=0.02,
                sticky_action_prob=0.05,
            )
        )

    random.shuffle(plan)
    return plan


def collect_dataset(
    *,
    start_episode_id: int = 1,
    plan: list[CollectionConfig] | None = None,
    dry_run: bool = True,
) -> list[dict[str, Any]]:
    plan = make_collection_plan() if plan is None else plan
    summaries = []

    for offset, config in enumerate(plan):
        episode_id = start_episode_id + offset
        episode_dir = RAW_EPISODES_DIR / f"ep_{episode_id:06d}"

        if episode_dir.exists() and not config.overwrite:
            summaries.append({"episode_id": episode_id, "run_type": config.run_type, "status": "skipped_exists"})
            continue

        if dry_run:
            summaries.append({"episode_id": episode_id, "run_type": config.run_type, "status": "planned"})
            continue

        summary = run_episode(episode_id, config)
        summary["run_type"] = config.run_type
        summary["status"] = "written"
        summaries.append(summary)

    return summaries


In [8]:
# Smoke test once make_env/load_policy are wired.
summary = run_episode(episode_id=1, config=CONFIG)
print("Smoke test episode summary:", summary)

# Verify recorded artifact
ep_dir = Path(summary["dir"])
assert ep_dir.exists(), f"Episode directory {ep_dir} does not exist!"

video_reader = imageio.get_reader(ep_dir / "frames.mkv")
frame_count = sum(1 for _ in video_reader)
video_reader.close()

with (ep_dir / "steps.jsonl").open() as f:
    step_count = sum(1 for _ in f)

print(f"Recorded frame count: {frame_count}")
print(f"Recorded step count: {step_count}")
assert frame_count == step_count, f"Frame count ({frame_count}) does not match step count ({step_count})"

# Preview the 80-episode collection plan without writing files.
plan_preview = collect_dataset(dry_run=True)
print("Plan preview (first 5):", plan_preview[:5])
print("Total episodes planned:", len(plan_preview))


Smoke test episode summary: {'episode_id': 1, 'steps': 381, 'reward': 121.95000456273556, 'dir': '/Users/soheilchavoshi/Projects/mario-world/data/raw/episodes/ep_000001'}
Recorded frame count: 381
Recorded step count: 381
Plan preview (first 5): [{'episode_id': 1, 'run_type': 'normal', 'status': 'planned'}, {'episode_id': 2, 'run_type': 'normal', 'status': 'planned'}, {'episode_id': 3, 'run_type': 'normal', 'status': 'planned'}, {'episode_id': 4, 'run_type': 'random_start_perturbed', 'status': 'planned'}, {'episode_id': 5, 'run_type': 'random_start_perturbed', 'status': 'planned'}]
Total episodes planned: 80
